# 预期ST因子测试

因子定义：连续两年净利润为负（老规则）

策略逻辑：
1. 每天从全市场筛选总市值最小的500只股票（尾部500）
2. 在尾部500中，筛选连续两年净利润为负的股票
3. 等权持有所有满足条件的股票
4. 每天调仓

In [1]:
from bigmodule import M, I
import dai
import pandas as pd


# @param(id="m5", name="initialize")
def m5_initialize_bigquant_run(context):
    from bigtrader.finance.commission import PerOrder
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))


# @param(id="m5", name="before_trading_start")
def m5_before_trading_start_bigquant_run(context, data):
    pass


# @param(id="m5", name="handle_tick")
def m5_handle_tick_bigquant_run(context, tick):
    pass


# @param(id="m5", name="handle_data")
def m5_handle_data_bigquant_run(context, data):
    # 每天都是调仓日
    today_df = context.data[context.data["date"] == data.current_dt.strftime("%Y-%m-%d")]
    target_instruments = set(today_df["instrument"])
    holding_instruments = set(context.get_account_positions().keys())

    # 卖出不在目标列表中的股票
    for instrument in holding_instruments - target_instruments:
        context.order_target_percent(instrument, 0)

    # 买入目标股票
    for i, x in today_df.iterrows():
        position = 0.0 if pd.isnull(x.position) else float(x.position)
        context.order_target_percent(x.instrument, position)


# @param(id="m5", name="handle_trade")
def m5_handle_trade_bigquant_run(context, trade):
    pass


# @param(id="m5", name="handle_order")
def m5_handle_order_bigquant_run(context, order):
    pass


# @param(id="m5", name="after_trading")
def m5_after_trading_bigquant_run(context, data):
    pass


# ========== 数据准备 ==========

# 预期ST因子SQL：连续两年净利润为负
# 使用 cn_stock_prefactors 表，需要确认字段名
stock_sql = """
SELECT
    date,
    instrument,
    market_cap,
    net_profit_ttm,
    -- 去年同期净利润（用于判断连续两年）
    m_lag(net_profit_ttm, 250) AS net_profit_ttm_ly
FROM cn_stock_prefactors
WHERE
    -- 非ST
    st_status = 0
    -- 非停牌
    AND suspended = 0
    -- 非北交所
    AND is_bz50 = 0
    -- 上市超过1年
    AND list_days > 365
QUALIFY
    -- 尾部500：按总市值排序取最小的500只
    ROW_NUMBER() OVER (PARTITION BY date ORDER BY market_cap ASC) <= 500
"""

print("正在查询数据...")
stock_data = dai.query(stock_sql, filters={"date": ["2020-01-01", "2026-12-31"]}).df()
print(f"查询完成，共 {len(stock_data)} 条记录")

# 筛选：连续两年净利润为负
# 条件：当年净利润 < 0 且 去年净利润 < 0
filtered_df = stock_data[
    (stock_data['net_profit_ttm'] < 0) & 
    (stock_data['net_profit_ttm_ly'] < 0)
].copy()
print(f"满足预期ST条件的记录数：{len(filtered_df)}")

# 计算每日持仓数量并分配等权仓位
daily_count = filtered_df.groupby('date')['instrument'].transform('count')
filtered_df['position'] = 1.0 / daily_count
filtered_df['score'] = -filtered_df['market_cap']  # 市值越小分数越高
filtered_df['score_rank'] = filtered_df.groupby('date')['market_cap'].rank(ascending=True).astype(int)

print(f"\n每日平均持仓数量：{daily_count.groupby(filtered_df['date']).first().mean():.1f}")

stock_data_ds = dai.DataSource.write_bdb(filtered_df)

# ========== 回测设置 ==========
start_date = '2021-01-01'
end_date = '2026-04-07'

m5 = M.bigtrader.v30(
    data=stock_data_ds,
    start_date=start_date,
    end_date=end_date,
    initialize=m5_initialize_bigquant_run,
    before_trading_start=m5_before_trading_start_bigquant_run,
    handle_tick=m5_handle_tick_bigquant_run,
    handle_data=m5_handle_data_bigquant_run,
    handle_trade=m5_handle_trade_bigquant_run,
    handle_order=m5_handle_order_bigquant_run,
    after_trading=m5_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="daily",
    product_type="股票",
    rebalance_period_type="交易日",
    rebalance_period_days="1",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="标准模式",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="open",
    order_price_field_sell="open",
    benchmark="沪深300指数",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="m5"
)

正在查询数据...


InvalidInputException: Invalid Input Error: Attempting to execute an unsuccessful or closed pending query result
Error: Binder Error: Referenced column market_cap not found in FROM clause and can't find in alias map.

In [ ]:
# ========== 导出交易记录CSV ==========
# 获取回测的交易记录
trades_df = m5.raw_perf.read()['transactions']

# 转换为类似果仁格式的CSV
# 果仁格式：股票代码,股票名,行业分类,二级行业,买入日期,卖出日期,买入价格(前复权),卖出价格(前复权),涨幅,相对收益

# 处理交易记录，配对买入卖出
output_records = []
holdings = {}  # instrument -> {buy_date, buy_price}

for idx, row in trades_df.iterrows():
    instrument = row['symbol']
    dt = pd.to_datetime(row['dt']).strftime('%Y-%m-%d')
    amount = row['amount']
    price = row['price']
    
    if amount > 0:  # 买入
        holdings[instrument] = {'buy_date': dt, 'buy_price': price}
    elif amount < 0 and instrument in holdings:  # 卖出
        buy_info = holdings.pop(instrument)
        pnl = (price - buy_info['buy_price']) / buy_info['buy_price']
        output_records.append({
            '股票代码': instrument.split('.')[0],
            '买入日期': buy_info['buy_date'],
            '卖出日期': dt,
            '买入价格(前复权)': round(buy_info['buy_price'], 2),
            '卖出价格(前复权)': round(price, 2),
            '涨幅': round(pnl, 4)
        })

output_df = pd.DataFrame(output_records)
output_df = output_df.sort_values('卖出日期', ascending=False)

# 保存CSV
output_path = './strategy/预期ST_bigquant交易记录.csv'
output_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"交易记录已保存到：{output_path}")
print(f"共 {len(output_df)} 条交易记录")
output_df.head(20)